## Pandas Analysis

### Project Overview
This notebook contains Pandas-based analysis of the TMDB Movie dataset.

### Python Library
Pandas

In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Connect to database
engine = create_engine("sqlite:///data/TMDB_database.db")

In [2]:
# Load all six cleaned tables into Pandas DataFrames
movies = pd.read_sql("SELECT * FROM movies", engine)
genres = pd.read_sql("SELECT * FROM genres", engine)
movie_genres = pd.read_sql("SELECT * FROM movie_genres", engine)
cast = pd.read_sql("SELECT * FROM cast", engine)
crew = pd.read_sql("SELECT * FROM crew", engine)
movie_keywords = pd.read_sql("SELECT * FROM movie_keywords", engine)

In [3]:
print(movies.shape)

(2503, 12)


## Q1. List the total number of unique movies, genres, cast members and crew members collected across all tables.

In [4]:
print("Unique movies:", movies["movie_id"].nunique())
print("Unique genres:", genres["genre_id"].nunique())
print("Unique cast members:", cast["person_id"].nunique())
print("Unique crew members:", crew["person_id"].nunique())

Unique movies: 2503
Unique genres: 19
Unique cast members: 11300
Unique crew members: 2848


**Insights** 
The dataset has 2,503 movies, 19 genres, 11,300 unique cast members and 2,848 unique crew members. 


## Q2. List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis.



In [5]:
Zero_budget = movies["budget"].isna() | (movies["budget"] == 0)
Zero_revenue = movies["revenue"].isna() | (movies["revenue"] == 0)
invalid_movies = movies[Zero_budget | Zero_revenue]
invalid_movies[["title", "budget", "revenue"]]

,title,budget,revenue
324,Zack Snyder's Justice League,70000000,0
340,Bird Box,19800000,0
403,365 Days,0,9458590
468,To All the Boys I've Loved Before,0,0
554,Prey,65000000,0
...,...,...,...
2469,Day Shift,100000000,0
2471,A Serbian Film,0,0
2480,Chungking Express,160000,0
2481,I'm Thinking of Ending Things,0,0


**Insights** 
133 movies have a zero budget or zero revenue. Zero means unknown, so I leave these movies out of profit and ROI calculations.

## Q3.List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.

In [6]:
valid_movies = movies[(movies["budget"] > 0) & (movies["revenue"] > 0)].copy()
print("Movies used for analysis:", len(valid_movies))

# Profit = revenue - budget
valid_movies["profit"] = valid_movies["revenue"] - valid_movies["budget"]

# ROI = profit / budget
valid_movies["roi"] = (valid_movies["profit"] / valid_movies["budget"]).round(2)

valid_movies[["title", "budget", "revenue", "profit", "roi"]]

Movies used for analysis: 2370


,title,budget,revenue,profit,roi
0,Interstellar,165000000,746606706,581606706,3.52
1,Inception,160000000,839030630,679030630,4.24
2,The Avengers,220000000,1518815515,1298815515,5.90
3,The Dark Knight,185000000,1004558444,819558444,4.43
4,Avatar,237000000,2923706026,2686706026,11.34
...,...,...,...,...,...
2498,Daybreakers,20000000,51417188,31417188,1.57
2499,Date Night,93214796,87595677,-5619119,-0.06
2500,Transformers,110098767,274746695,164647928,1.50
2501,The SpongeBob Movie: Sponge Out of Water,39284878,570657513,531372635,13.53


**Insights** 
2,370 movies have valid budget and revenue, so profit and ROI are calculated only for them.

## Q4.List the top 15 movies by return on investment, considering only movies with a budget of at least $1 million.



In [7]:
top15 = valid_movies[valid_movies["budget"] >= 1000000].sort_values("roi",ascending=False)

top15[["title", "budget", "revenue", "roi"]].head(15)

,title,budget,revenue,roi
1767,Dragon Ball Super: Broly,1000000,125002821,124.00
535,Snow White and the Seven Dwarfs,1488423,184925486,123.24
1791,The Rocky Horror Picture Show,1400000,171181400,121.27
467,Rocky,1000000,117253345,116.25
1227,Gone with the Wind,4000000,402352579,99.59
716,The Jungle Book,4000000,378000000,93.50
635,Cinderella,2900000,263600000,89.90
372,Saw,1200000,104045735,85.70
719,One Hundred and One Dalmatians,3600000,303000000,83.17
263,E.T. the Extra-Terrestrial,10500000,797307407,74.93


**Insights** 
Dragon Ball Super: Broly has the highest ROI (124x). Most top ROI movies had small budgets, so low-budget films can earn the most per dollar.

## Q5.List the number of movies released per year, and identify which year had the highest movie output in the dataset.



In [8]:
movies["release_year"] = pd.to_datetime(movies["release_date"]).dt.year

movies_per_year = movies["release_year"].value_counts().sort_index()

movies_per_year

release_year
1921     1
1922     1
1927     1
1931     2
1936     1
        ..
2022    65
2023    62
2024    43
2025    26
2026     8
Name: count, Length: 89, dtype: int64

In [9]:
highest_year = movies_per_year.idxmax()
highest_count = movies_per_year.max()

print("Year with highest movie releases:", highest_year)
print("Number of movies released:", highest_count)

Year with highest movie releases: 2016
Number of movies released: 121


**Insights** 
2016 had the most movies (121). The low counts for 2025 and 2026 are because those years are only partly covered in the dataset.

## Q6.List all movies whose runtime is a statistical outlier compared to the rest of the dataset.



In [10]:
q1 = movies["runtime"].quantile(0.25)
q3 = movies["runtime"].quantile(0.75)
iqr = q3 - q1

outliers = movies[(movies["runtime"] < q1 - 1.5 * iqr) | (movies["runtime"] > q3 + 1.5 * iqr)]
outliers[["title", "runtime"]]

,title,runtime
0,Interstellar,169
16,Avengers: Endgame,181
18,The Lord of the Rings: The Fellowship of the Ring,179
19,Titanic,194
20,The Lord of the Rings: The Return of the King,201
21,The Wolf of Wall Street,180
28,The Lord of the Rings: The Two Towers,179
38,The Godfather,175
76,The Hobbit: An Unexpected Journey,169
79,The Green Mile,189


**Insights** 
Some movies are much longer than normal, like Titanic (194 min). I keep them because they are real movies, not errors.

 ## Q7.List each movie's primary genre (the first genre listed) and compare average popularity across primary genres.



In [11]:
first_genre = movie_genres.drop_duplicates("movie_id")
data = movies.merge(first_genre, on="movie_id").merge(genres, on="genre_id")

print(data[["title", "genre_name"]].head(10))
data.groupby("genre_name")["popularity"].mean().round(2).sort_values(ascending=False)

                      title       genre_name
0              Interstellar        Adventure
1                 Inception           Action
2              The Avengers  Science Fiction
3           The Dark Knight           Action
4                    Avatar           Action
5                  Deadpool           Action
6                Fight Club            Drama
7    Avengers: Infinity War        Adventure
8  The Shawshank Redemption            Drama
9              Pulp Fiction         Thriller


genre_name
Science Fiction    21.26
Music              18.32
Animation          16.81
Family             16.71
Action             16.39
War                15.47
Horror             14.85
Adventure          14.79
Fantasy            13.02
Mystery            12.43
Western            12.32
Thriller           11.29
Drama              11.28
Crime              10.74
History            10.41
Comedy              9.98
Romance             9.98
Documentary         7.73
Name: popularity, dtype: float64

**Insights** 
Science Fiction has the highest average popularity (21.26). Documentary has the lowest (7.73).

## Q8.List the top 10 most prolific actors by number of movies, along with their average movie rating across those movies.



In [12]:
data = cast.merge(movies[["movie_id", "vote_average"]], on="movie_id")

top10 = data.groupby("actor_name").agg(movie_count=("movie_id", "count"), avg_rating=("vote_average", "mean")).round(2)
top10.sort_values("movie_count", ascending=False).head(10)

,movie_count,avg_rating
actor_name,,
Samuel L. Jackson,39,6.94
Robert De Niro,38,7.23
Brad Pitt,38,7.26
Johnny Depp,37,6.86
Tom Hanks,33,7.29
Willem Dafoe,32,7.11
Mark Wahlberg,32,6.63
Scarlett Johansson,32,7.06
Tom Cruise,31,7.04


**Insights** 
Samuel L. Jackson has appeared in the most movies (39). Tom Hanks has the best average rating (7.29) among the top 10 actors.

## Q9.List any movie titles that appear more than once in the dataset, along with their release years, to check for duplicates or remakes.



In [13]:
dups = movies[movies.duplicated("title", keep=False)].sort_values("title")
print(len(dups))
dups[["title", "release_date"]]

100


,title,release_date
878,A Nightmare on Elm Street,1984-11-09
1928,A Nightmare on Elm Street,2010-04-30
261,Aladdin,1992-11-25
339,Aladdin,2019-05-22
166,Alice in Wonderland,2010-03-03
...,...,...
1737,The Thing,2011-10-12
744,Total Recall,1990-06-01
860,Total Recall,2012-08-02
252,Transformers,2007-06-27


**Insights** 
57 titles appear more than once, but each has different release years. 
So they are remakes, not duplicates, for example Aladdin (1992 and 2019).

## Q10.List movies whose popularity score and vote count don't follow the general pattern between the two — i.e., unusually high popularity with a low vote count, or vice versa.



In [14]:
h_pop = movies["popularity"].quantile(0.90)
l_pop = movies["popularity"].quantile(0.25)
l_votes = movies["vote_count"].quantile(0.25)
h_votes = movies["vote_count"].quantile(0.75)

a = movies[(movies["popularity"] > h_pop) & (movies["vote_count"] < l_votes)]
print("High popularity, low votes:", len(a))
a[["title", "popularity", "vote_count"]]

High popularity, low votes: 18


,title,popularity,vote_count
1889,How to Train Your Dragon,35.4872,3008
1897,Mission: Impossible - The Final Reckoning,31.1708,3001
1898,Final Destination Bloodlines,34.1109,3001
1923,Flow,23.5244,2956
1998,Ballerina,23.5962,2836
2022,The Housemaid,41.4337,2789
2049,Malena,23.7035,2753
2050,28 Years Later,27.4820,2752
2065,Mufasa: The Lion King,24.2199,2728
2248,Backrooms,258.4743,2483


In [15]:
low_pop = movies["popularity"].quantile(0.25)
high_votes = movies["vote_count"].quantile(0.75)

b = movies[(movies["popularity"] < low_pop) & (movies["vote_count"] > high_votes)]
print("Low popularity, high votes:", len(b))
b[["title", "popularity", "vote_count"]]

Low popularity, high votes: 33


,title,popularity,vote_count
101,The Silence of the Lambs,7.0584,18235
195,Moana,7.1410,13886
205,X-Men: First Class,1.2614,13680
244,Home Alone,1.8114,12658
252,Transformers,6.5902,12381
254,X-Men,1.2978,12340
290,Jumanji,5.2543,11431
294,X-Men Origins: Wolverine,2.3595,11379
303,Fast & Furious 6,1.4240,11275
311,The Theory of Everything,7.3158,11195


**Insights** 
Old movies like Home Alone have many votes but low popularity. 
This shows popularity means current attention, and vote count means total audience over time.